# Lab 07: Preprocessing

**Week:** 4
**Type:** Independent
**Corresponding working sessions:** `working-sessions/preprocessing/`
**Estimated time:** 60–75 minutes
**Dataset:** Heart Disease (Cleveland)

---

### Learning Objectives

By the end of this lab you will be able to:

- **Identify** missing values and select an appropriate imputation strategy
- **Encode** a nominal categorical feature using one-hot encoding
- **Apply** StandardScaler to continuous features to normalize their scale
- **Split** data into training and test sets in the correct order
- **Explain** why preprocessing steps must be fit on the training set only

---

### Context

You are continuing with the Heart Disease dataset from Lab 05.
The goal is no longer exploration — it is preparation.

Before any machine learning model can be trained, the data must be clean, consistently
encoded, and scaled. In this lab you will work through the four core preprocessing steps
in the order a working data scientist would apply them:

1. Handle missing values
2. Encode categorical features
3. Split into train and test sets
4. Scale continuous features — using only the training data

Run the cells in order. Each section builds on the one before it.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from tkh_utils import (
    PALETTE, FONT, base_layout,
    check_answer, make_answer_key, make_grading_summary,
    load_heart_disease
)

In [ ]:
_ak = make_answer_key({
    'q1': 'A',
    'q2': 'C',
    'q3': 'B',
    'q4': 'B',
})

X, y = load_heart_disease()
df = X.copy()
df['target'] = y

print(f"Dataset loaded: {df.shape[0]} patients, {df.shape[1]} columns")
df.head()

---

**Column reference:**

| Column | Description |
|--------|-------------|
| `age` | Age in years |
| `sex` | Sex (1 = male, 0 = female) |
| `cp` | Chest pain type (0 = typical angina, 1 = atypical angina, 2 = non-anginal pain, 3 = asymptomatic) |
| `trestbps` | Resting blood pressure (mm Hg) |
| `chol` | Serum cholesterol (mg/dl) |
| `fbs` | Fasting blood sugar > 120 mg/dl (1 = true, 0 = false) |
| `restecg` | Resting ECG results (0–2) |
| `thalach` | Maximum heart rate achieved |
| `exang` | Exercise-induced angina (1 = yes, 0 = no) |
| `oldpeak` | ST depression induced by exercise |
| `slope` | Slope of peak exercise ST segment (0–2) |
| `ca` | Number of major vessels colored by fluoroscopy (0–3) |
| `thal` | Thalassemia type (1 = normal, 2 = fixed defect, 3 = reversible defect) |
| `target` | Outcome: 1 = heart disease detected, 0 = not detected |

In [ ]:
# Pre-filled — run this cell to simulate missing values
np.random.seed(7)
ca_idx = np.random.choice(df.index, size=25, replace=False)
thal_idx = np.random.choice(df.index, size=15, replace=False)
df.loc[ca_idx, 'ca'] = np.nan
df.loc[thal_idx, 'thal'] = np.nan

print("Missing values injected:")
print(f"  ca:   {df['ca'].isnull().sum()} missing ({df['ca'].isnull().mean()*100:.1f}%)")
print(f"  thal: {df['thal'].isnull().sum()} missing ({df['thal'].isnull().mean()*100:.1f}%)")

---

## Section A — Conceptual Questions

For each question below, replace `"___"` with your answer (`"A"`, `"B"`, `"C"`, or `"D"`).
Run the cell after filling in your answer — you will get immediate feedback.

In [ ]:
# Question 1
# What does StandardScaler do to a numeric feature?
#
#   A) Subtracts the mean and divides by the standard deviation (produces Z-scores)
#   B) Scales all values to a range between 0 and 1
#   C) Replaces each value with its percentile rank across the dataset
#   D) Removes outliers by clipping values beyond 3 standard deviations

q1_answer = "___"  # Replace with A, B, C, or D

assert q1_answer != "___", "Don't forget to fill in your answer."
assert check_answer(q1_answer, _ak['q1']), \
    "Not quite — think about what 'standard' means in StandardScaler. " \
    "Revisit working-sessions/preprocessing/04_feature_scaling.ipynb"
print("\u2713 Question 1 correct!")

In [ ]:
# Question 2
# One-hot encoding is most appropriate for which type of feature?
#
#   A) A continuous numeric feature like age or cholesterol
#   B) An ordinal feature where the category order carries real meaning (e.g., low/medium/high)
#   C) A nominal categorical feature where categories have no natural order (e.g., chest pain type)
#   D) A binary feature already encoded as 0 and 1

q2_answer = "___"  # Replace with A, B, C, or D

assert q2_answer != "___", "Don't forget to fill in your answer."
assert check_answer(q2_answer, _ak['q2']), \
    "Not quite — think about which type of feature has no meaningful numeric ordering. " \
    "Revisit working-sessions/preprocessing/03_encoding_categorical_variables.ipynb"
print("\u2713 Question 2 correct!")

In [ ]:
# Question 3
# Which of the following is an example of data leakage?
#
#   A) Adding a new feature engineered entirely from training data
#   B) Fitting your StandardScaler on the full dataset before splitting into train and test
#   C) Using stratified sampling to preserve class balance in the split
#   D) Dropping rows with missing values before training

q3_answer = "___"  # Replace with A, B, C, or D

assert q3_answer != "___", "Don't forget to fill in your answer."
assert check_answer(q3_answer, _ak['q3']), \
    "Not quite — data leakage means information from outside the training set influences the model. " \
    "Revisit working-sessions/preprocessing/08_data_leakage.ipynb"
print("\u2713 Question 3 correct!")

In [ ]:
# Question 4
# A numeric column has several extreme high values. Why is the median a better choice
# than the mean for imputing missing values in this column?
#
#   A) The mean is computationally slower and should be avoided for large datasets
#   B) The median is resistant to outliers and better represents a typical value
#      in a skewed distribution
#   C) The mean always overestimates the true value when extreme values are present
#   D) The median guarantees the imputed value already exists in the dataset

q4_answer = "___"  # Replace with A, B, C, or D

assert q4_answer != "___", "Don't forget to fill in your answer."
assert check_answer(q4_answer, _ak['q4']), \
    "Not quite — think about what happens to the mean when a few extreme values are present. " \
    "Revisit working-sessions/preprocessing/02_handling_missing_data.ipynb"
print("\u2713 Question 4 correct!")

---

## Section B — Code Exercises

Fill in each `___` to complete the code. Run the cell to check your work.

### Exercise B1 — Imputing missing values

The two columns with missing values are `ca` and `thal`.

- `ca` (number of major vessels) has a skewed distribution — use the **median**.
- `thal` (thalassemia type) is a categorical feature stored as a number — use the **mode**.

Your tasks:
- Store the fill value for `ca` as `ca_fill_value`
- Store the fill value for `thal` as `thal_fill_value`
- Fill both columns and verify no missing values remain

*Reference: `working-sessions/preprocessing/02_handling_missing_data.ipynb`*

In [ ]:
ca_fill_value = ___             # median of the 'ca' column (ignoring NaNs)

thal_fill_value = ___           # mode of the 'thal' column — hint: .mode()[0]

df['ca']   = df['ca'].fillna(___)
df['thal'] = df['thal'].fillna(___)

remaining_missing = df[['ca', 'thal']].isnull().sum().sum()

# --- checks ---
assert ca_fill_value == 0.0, \
    f"ca_fill_value should be 0.0 (the median) — got {ca_fill_value}"
assert thal_fill_value == 2.0, \
    f"thal_fill_value should be 2.0 (the mode) — got {thal_fill_value}"
assert remaining_missing == 0, \
    "Both columns should have 0 missing values after filling"
print(f"\u2713 Imputation complete")
print(f"  ca   filled with median:  {ca_fill_value}")
print(f"  thal filled with mode:    {thal_fill_value}")
print(f"  Remaining missing: {remaining_missing}")

### Exercise B2 — One-hot encoding

`cp` (chest pain type) has four categories: 0, 1, 2, and 3. These are labels — not numbers
you can do arithmetic on. Treating them as numeric would imply that type 3 is "three times"
type 1, which is meaningless.

One-hot encoding replaces a single column with one binary column per category.

Your tasks:
- Use `pd.get_dummies()` to one-hot encode the `cp` column, storing the result as `df_encoded`
- Count how many columns `df_encoded` has (store as `n_cols_after`)

*Reference: `working-sessions/preprocessing/03_encoding_categorical_variables.ipynb`*

In [ ]:
n_cols_before = df.shape[1]

df_encoded = pd.get_dummies(___, columns=['cp'], drop_first=False)

n_cols_after = ___              # number of columns in df_encoded

# --- checks ---
assert n_cols_before == 14, \
    f"Expected 14 columns before encoding — got {n_cols_before}"
assert n_cols_after == 17, \
    f"Expected 17 columns after encoding — got {n_cols_after}. " \
    f"Hint: 14 - 1 (cp removed) + 4 (cp_0.0 through cp_3.0 added) = 17"
assert 'cp_0.0' in df_encoded.columns and 'cp_3.0' in df_encoded.columns, \
    "Columns cp_0.0 through cp_3.0 should exist in df_encoded"
print(f"\u2713 One-hot encoding complete")
print(f"  Columns before: {n_cols_before}")
print(f"  Columns after:  {n_cols_after}")
print(f"  New columns: {[c for c in df_encoded.columns if c.startswith('cp_')]}")

### Exercise B3 — Scaling continuous features

Scaling must happen *after* the train-test split. The scaler must be **fit on the training
set only** — then used to transform both the training and test sets.

The five continuous features that need scaling are: `age`, `trestbps`, `chol`, `thalach`, `oldpeak`.

The setup cell below has already created the split. Your tasks:
- Call `fit_transform` on `X_b3_train` to fit the scaler and scale it in one step
- Call `transform` (not `fit_transform`) on `X_b3_test`
- Verify that the training set means are approximately zero

*Reference: `working-sessions/preprocessing/04_feature_scaling.ipynb` and `05_feature_scaling_in_practice.ipynb`*

In [ ]:
# Pre-filled — run this cell to create the train-test split for Exercise B3
X_b3 = df_encoded.drop('target', axis=1)
y_b3 = df_encoded['target']

X_b3_train, X_b3_test, y_b3_train, y_b3_test = train_test_split(
    X_b3, y_b3, test_size=0.2, random_state=42, stratify=y_b3
)
print(f"Train: {len(X_b3_train)} samples  |  Test: {len(X_b3_test)} samples")

In [ ]:
cont_cols = ['age', 'trestbps', 'chol', 'thalach', 'oldpeak']

scaler = StandardScaler()

X_b3_train[cont_cols] = ___   # fit the scaler on X_b3_train and transform it
X_b3_test[cont_cols]  = ___   # transform X_b3_test using the already-fitted scaler

max_abs_mean = abs(X_b3_train[cont_cols].mean()).max()

# --- checks ---
assert max_abs_mean < 0.01, \
    f"Training means should be ~0 after scaling — max absolute mean was {max_abs_mean:.4f}. " \
    "Did you call fit_transform on X_b3_train?"
print(f"\u2713 Scaling complete — training means are ~0")
print(f"  Max absolute mean across continuous cols: {max_abs_mean:.6f}")
print()
print("Mean of each continuous feature in the training set after scaling:")
print(X_b3_train[cont_cols].mean().round(6).to_string())

---

## Section C — Applied Preprocessing Pipeline

Apply the full preprocessing pipeline from scratch, in the correct order.
The setup cell below provides a fresh copy of the data with the same missing values
already injected. Fill in each `___` and run all cells in order.

In [ ]:
# Pre-filled — fresh copy with missing values, same seed as above
np.random.seed(7)
X_raw, y_raw = load_heart_disease()
df_c = X_raw.copy()
df_c['target'] = y_raw
df_c.loc[np.random.choice(df_c.index, size=25, replace=False), 'ca'] = np.nan
df_c.loc[np.random.choice(df_c.index, size=15, replace=False), 'thal'] = np.nan
print(f"Fresh data: {df_c.shape[0]} rows, {df_c.isnull().sum().sum()} missing values")

### C1 — Full pipeline

Work through all five steps in order. Each step has a check — fix any failures before
moving on.

In [ ]:
# Step 1: impute missing values
df_c['ca']   = df_c['ca'].fillna(___)     # median
df_c['thal'] = df_c['thal'].fillna(___)   # mode

assert df_c.isnull().sum().sum() == 0, "Missing values still remain — check your fillna calls"
print(f"\u2713 Step 1: imputation complete")

# Step 2: one-hot encode cp
df_c = pd.get_dummies(___, columns=['cp'], drop_first=False)

assert df_c.shape[1] == 17, \
    f"Expected 17 columns after encoding — got {df_c.shape[1]}"
print(f"\u2713 Step 2: cp encoded — shape is now {df_c.shape}")

# Step 3: separate features and target
X_c = df_c.drop('target', axis=1)
y_c = df_c['target']

# Step 4: train-test split — BEFORE scaling
X_c_train, X_c_test, y_c_train, y_c_test = train_test_split(
    X_c, y_c, test_size=___, random_state=42, stratify=y_c
)

assert len(X_c_train) == 242, f"Expected 242 training samples — got {len(X_c_train)}"
assert len(X_c_test)  == 61,  f"Expected 61 test samples — got {len(X_c_test)}"
print(f"\u2713 Step 3: split — {len(X_c_train)} train, {len(X_c_test)} test")

# Step 5: scale continuous features — fit on training data ONLY
cont_cols_c = ['age', 'trestbps', 'chol', 'thalach', 'oldpeak']
scaler_c = StandardScaler()

X_c_train[cont_cols_c] = ___   # fit_transform on training set
X_c_test[cont_cols_c]  = ___   # transform only on test set

assert abs(X_c_train[cont_cols_c].mean()).max() < 0.01, \
    "Training means should be ~0 — did you use fit_transform on X_c_train?"
print(f"\u2713 Step 4: scaling complete — training means are ~0")
print()
print(f"Pipeline complete. Ready for modeling.")
print(f"  Train: {X_c_train.shape[0]} samples \u00d7 {X_c_train.shape[1]} features")
print(f"  Test:  {X_c_test.shape[0]} samples \u00d7 {X_c_test.shape[1]} features")

### C2 — Visualize the scaling effect

Create a side-by-side histogram comparing the `age` distribution in the training set
**before** and **after** scaling. Label both axes and add a title.

The pre-filled line gives you the unscaled ages for the training rows.

*Reference: `working-sessions/preprocessing/04_feature_scaling.ipynb`*

In [ ]:
age_unscaled = X_raw.loc[X_c_train.index, 'age']   # pre-filled

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

# left: unscaled
sns.histplot(___, bins=20, ax=axes[0])
axes[0].set_title('Age — Before Scaling')
axes[0].set_xlabel('Age (years)')
axes[0].set_ylabel('Count')

# right: scaled
sns.histplot(___, bins=20, ax=axes[1])
axes[1].set_title('Age — After StandardScaler')
axes[1].set_xlabel('Z-score')
axes[1].set_ylabel('Count')

plt.suptitle('Effect of StandardScaler on the age distribution', y=1.02)
plt.tight_layout()
plt.show()

print("Notice: the shape of the distribution is identical — only the axis scale has changed.")

---

## Section D — Interpretation

These questions are for reflection. Edit the markdown cells below each question to write
your response. There are no wrong answers — we are looking for thoughtful engagement
with the concepts. Your instructor may review these.

**Question D1**

In Exercise B3 and Section C, you used `fit_transform` on the training set and `transform`
(without `fit`) on the test set.

In plain language, explain why this matters. What would go wrong if you called `fit_transform`
on the full dataset before splitting — or if you called `fit_transform` on both the training
and test sets separately?

*Your response here...*

**Question D2**

Look at the columns `sex`, `fbs`, and `exang`. Each is a binary feature encoded as 0 or 1.

You scaled the five continuous features but left these binary columns untouched. Was that
the right call? What happens to a binary column when you apply StandardScaler to it?
Walk through the reasoning, not just the conclusion.

*Your response here...*

**Question D3**

You applied one-hot encoding to `cp` (chest pain type) even though it was stored as
0.0, 1.0, 2.0, 3.0 — floating-point numbers that look numeric.

Explain why leaving `cp` as a raw numeric feature could mislead a machine learning model.
What assumption would the model be making, and why is that assumption wrong for a feature
like chest pain type?

*Your response here...*

---

## Self-Grading Summary

Run the cell below when you are ready to check your Section A score.
Fix any failing checks above and re-run until everything passes.

In [ ]:
make_grading_summary([
    (q1_answer, _ak['q1'], "Q1: What StandardScaler does"),
    (q2_answer, _ak['q2'], "Q2: When to use one-hot encoding"),
    (q3_answer, _ak['q3'], "Q3: Data leakage example"),
    (q4_answer, _ak['q4'], "Q4: Mean vs median imputation"),
], total=4)
print("Section B & C: check \u2713 marks above")
print("Section D: for your own reflection")

*Next up: supervised learning — where a preprocessed dataset like this one gets handed off to a model.*